# Customer Churn Prediction

This notebook predicts customer churn using three ML models: Logistic Regression, Random Forest, and XGBoost.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
np.random.seed(42)
n_samples = 1000

data = {
    'CustomerID': [f'C{str(i).zfill(4)}' for i in range(1, n_samples+1)],
    'Age': np.random.randint(18, 70, n_samples),
    'Tenure_Months': np.random.randint(1, 72, n_samples),
    'Monthly_Charges': np.round(np.random.uniform(20, 120, n_samples), 2),
    'Total_Charges': np.zeros(n_samples),
    'Contract_Type': np.random.choice(['Month-to-Month', 'One_Year', 'Two_Year'], n_samples),
    'Payment_Method': np.random.choice(['Electronic_Check', 'Mailed_Check', 'Bank_Transfer', 'Credit_Card'], n_samples),
    'Num_Services': np.random.randint(1, 8, n_samples),
    'Support_Calls': np.random.randint(0, 10, n_samples),
    'Online_Backup': np.random.choice(['Yes', 'No'], n_samples),
    'Tech_Support': np.random.choice(['Yes', 'No'], n_samples)
}

df = pd.DataFrame(data)
df['Total_Charges'] = np.round(df['Tenure_Months'] * df['Monthly_Charges'], 2)
df['Churn'] = np.where(
    (df['Support_Calls'] > 5) | 
    ((df['Monthly_Charges'] > 80) & (df['Contract_Type'] == 'Month-to-Month')) |
    ((df['Tenure_Months'] < 12) & (df['Support_Calls'] > 3)),
    1, 0
)
df['Churn'] = np.where(np.random.random(n_samples) < 0.1, 1 - df['Churn'], df['Churn'])
df['Churn'] = df['Churn'].astype(int)

In [ ]:
print('=== Dataset Info ===')
print(df.info())
print('\n=== Statistical Summary ===')
print(df.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(df['Churn'].value_counts(), labels=['Not Churned', 'Churned'], autopct='%1.1f%%')
axes[0].set_title('Churn Distribution')
churn_by_contract = df.groupby('Contract_Type')['Churn'].mean()
axes[1].bar(churn_by_contract.index, churn_by_contract.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1].set_title('Churn Rate by Contract Type')
axes[1].set_ylabel('Churn Rate')
axes[1].set_xticklabels(churn_by_contract.index, rotation=45, ha='right')
for i, v in enumerate(churn_by_contract.values):
    axes[1].text(i, v + 0.02, f'{v:.2%}', ha='center')
plt.tight_layout()
plt.show()

In [ ]:
df['Contract_Type_Encoded'] = df['Contract_Type'].map({'Month-to-Month': 0, 'One_Year': 1, 'Two_Year': 2})
df['Payment_Method_Encoded'] = df['Payment_Method'].map({'Electronic_Check': 0, 'Mailed_Check': 1, 'Bank_Transfer': 2, 'Credit_Card': 3})
df['Online_Backup_Binary'] = df['Online_Backup'].map({'Yes': 1, 'No': 0})
df['Tech_Support_Binary'] = df['Tech_Support'].map({'Yes': 1, 'No': 0})
features = ['Age', 'Tenure_Months', 'Monthly_Charges', 'Total_Charges', 'Contract_Type_Encoded', 'Payment_Method_Encoded', 'Num_Services', 'Support_Calls', 'Online_Backup_Binary', 'Tech_Support_Binary']
X = df[features].values
y = df['Churn'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Train churn rate: {y_train.mean():.2%}')
print(f'Test churn rate: {y_test.mean():.2%}')

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Features scaled successfully!')

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'classification_report': classification_report(y_test, y_pred)
    }

print('\n- Logistic Regression')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_results = evaluate_model(lr_model, X_test_scaled, y_test, 'Logistic Regression')

In [ ]:
print('\n- Random Forest Classifier')
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
rf_results = evaluate_model(rf_model, X_test, y_test, 'Random Forest')

In [ ]:
print('\n- XGBoost Classifier')
xgb_model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_results = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

In [ ]:
results = [lr_results, rf_results, xgb_results]
results_df = pd.DataFrame(results)
print('\n=== Model Comparison ===')
print(results_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
metrics = ['accuracy', 'precision', 'recall', 'f1']
colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12']
for i, metric in enumerate(metrics):
    row, col = i // 2, i % 2
    values = [r[metric] for r in results]
    bars = axes[row, col].bar(results_df['model'], values, color=colors[i])
    axes[row, col].set_ylabel(metric.title())
    axes[row, col].set_title(f'{metric.title()} by Model')
    axes[row, col].set_ylim(0, 1)
    for bar, val in zip(bars, values):
        axes[row, col].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
best_model_idx = results_df['roc_auc'].idxmax()
best_model = results_df.iloc[best_model_idx]['model']
best_auc = results_df.iloc[best_model_idx]['roc_auc']
print('\n=== FINAL RESULTS ===')
print(f'Best Model: {best_model}')
print(f'ROC AUC Score: {best_auc:.4f}')
print('\nAll three models successfully trained and evaluated!')